In [35]:
import torch
import numpy as np
from datasets import load_dataset, Audio
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
import evaluate

In [ ]:
# Load dataset (english)
minds14 = load_dataset("PolyAI/minds14", "en-US")
minds14 = minds14.cast_column("audio", Audio(sampling_rate=16000))

print(minds14)

DatasetDict({
    train: Dataset({
        features: ['path', 'audio', 'transcription', 'english_transcription', 'intent_class', 'lang_id'],
        num_rows: 563
    })
})


In [37]:
audio_input = minds14["train"][0]["audio"]  # first decoded audio sample
intent_class = minds14["train"][0]["intent_class"]  # first transcription
intent = minds14["train"].features["intent_class"].names[intent_class]

print(audio_input)
print(intent_class)
print(intent)

{'path': 'C:\\Users\\bertr\\.cache\\huggingface\\datasets\\downloads\\extracted\\ac262d1a09732a7cc73c079a12a1916cd94f646ac6664cde792d04eb267f35ab\\en-US~JOINT_ACCOUNT\\602ba55abb1e6d0fbce92065.wav', 'array': array([ 1.70561689e-05,  2.18727393e-04,  2.28099860e-04, ...,
        3.43842403e-05, -5.96366226e-06, -1.76846370e-05]), 'sampling_rate': 16000}
11
joint_account


In [ ]:
if "validation" not in minds14:
    # (10% for validation)
    train_test_split = minds14["train"].train_test_split(test_size=0.1, seed=42)
    minds14["train"] = train_test_split["train"]
    minds14["validation"] = train_test_split["test"]
    print("Created validation split from train data")
    print(f"Updated splits: {list(minds14.keys())}")
    print(f"Train size: {len(minds14['train'])}, Validation size: {len(minds14['validation'])}")

Created validation split from train data
Updated splits: ['train', 'validation']
Train size: 506, Validation size: 57


In [ ]:
model_name = "openai/whisper-tiny"
processor = WhisperProcessor.from_pretrained(model_name)
model = WhisperForConditionalGeneration.from_pretrained(model_name)

model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(task="transcribe")
model.config.suppress_tokens = []

In [ ]:
def prepare_dataset(examples):
    audio_arrays = [example["array"] for example in examples["audio"]]
    
    inputs = processor.feature_extractor(
        audio_arrays, 
        sampling_rate=16000,
        return_tensors=None
    )
    
    transcriptions = []
    for i in range(len(examples["audio"])):
        if "transcription" in examples and examples["transcription"][i]:
            transcriptions.append(examples["transcription"][i])
        else:
            transcriptions.append(examples["english_transcription"][i])
    
    labels = processor.tokenizer(transcriptions, return_tensors=None)
    
    result = {}
    result.update(inputs)
    result["labels"] = labels["input_ids"]
    
    return result

In [ ]:
tokenized_minds14 = minds14.map(
    prepare_dataset, 
    batched=True, 
    batch_size=4,
    remove_columns=minds14["train"].column_names,
)

Map: 100%|██████████| 57/57 [00:01<00:00, 41.85 examples/s]


In [ ]:
class WhisperDataCollator:
    def __init__(self, processor, model):
        self.processor = processor
        self.model = model
        
    def __call__(self, features):
        input_features_list = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features_list, return_tensors="pt")
        
        label_ids = [feature["labels"] for feature in features]
        
        labels_batch = self.processor.tokenizer.pad(
            {"input_ids": label_ids}, 
            return_tensors="pt"
        )
        
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch["input_ids"] == self.processor.tokenizer.pad_token_id, -100
        )
        
        batch["labels"] = labels
        
        return batch

data_collator = WhisperDataCollator(processor, model)

In [ ]:
metric = evaluate.load("wer")

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    
    wer = metric.compute(predictions=pred_str, references=label_str)
    
    return {"wer": wer}

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-minds14",
    eval_strategy="steps",
    eval_steps=1000,
    learning_rate=1e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    warmup_steps=500,
    save_steps=1000,
    logging_steps=250,
    push_to_hub=False,
    generation_max_length=225,
    predict_with_generate=True,
    fp16=True,
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_minds14["train"],
    eval_dataset=tokenized_minds14["validation"],
    compute_metrics=compute_metrics,
    tokenizer=processor.tokenizer,
    data_collator=data_collator,
)

C:\Users\bertr\AppData\Local\Temp\ipykernel_1876\1139338546.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [47]:
trainer.train()

Step,Training Loss,Validation Loss


c:\Users\bertr\anaconda3\envs\speech_recognition\lib\site-packages\transformers\modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 448, 'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=96, training_loss=2.10210911432902, metrics={'train_runtime': 1582.6149, 'train_samples_per_second': 0.959, 'train_steps_per_second': 0.061, 'total_flos': 3.737146761216e+16, 'train_loss': 2.10210911432902, 'epoch': 3.0})

In [48]:
model_path = "./whisper-minds14-finetuned"
model.save_pretrained(model_path)
processor.save_pretrained(model_path)
print(f"Model saved to {model_path}")

Model saved to ./whisper-minds14-finetuned


In [ ]:
# testing with a sample set
def transcribe_sample(audio_input):
    input_features = processor(
        audio_input["array"], 
        sampling_rate=16000, 
        return_tensors="pt"
    ).input_features
    
    predicted_ids = model.generate(input_features)
    
    transcription = processor.batch_decode(
        predicted_ids, 
        skip_special_tokens=True
    )[0]
    
    return transcription

In [ ]:
# choose sample
test_sample = minds14["validation"][0]
audio_input = test_sample["audio"]
intent_class = test_sample["intent_class"]
intent = minds14["validation"].features["intent_class"].names[intent_class]

print("\nTranscribing a test sample...")
print(f"True intent: {intent}")

predicted_transcription = transcribe_sample(audio_input)
print(f"Predicted transcription: {predicted_transcription}")

if "transcription" in test_sample and test_sample["transcription"]:
    print(f"Original transcription: {test_sample['transcription']}")
elif "english_transcription" in test_sample:
    print(f"Original English transcription: {test_sample['english_transcription']}")

Due to a bug fix in https://github.com/huggingface/transformers/pull/28687 transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English.This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`.



Transcribing a test sample...
True intent: cash_deposit


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Predicted transcription:  Hi hello I just need to deposit money into my account please.
Original transcription: hi hello I just need to deposit money into my account please
